# Dataset Exploration & Baseline Model Training

Ce notebook contient l'exploration du dataset et l'entraînement d'un modèle de régression logistique comme baseline.

## Structure
1. Setup & Imports
2. Phase 1: Exploration & Analyse
3. Phase 2: Nettoyage & Préparation
4. Phase 3: Feature Engineering
5. Phase 4: Baseline Model - Régression Logistique
6. Phase 5: Analyse des Résultats & Optimisation


## 1. Setup & Imports


In [ ]:
# Standard libraries
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP & ML
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    precision_recall_fscore_support
)

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
DATA_DIR = '../dataset'
print("✅ Imports réussis")


## 2. Phase 1: Exploration & Analyse du Dataset

### 2.1 Chargement des Données


In [ ]:
# Charger les différents datasets
df_original = pd.read_csv(f'{DATA_DIR}/travel_dataset.csv')
df_extended = pd.read_csv(f'{DATA_DIR}/extended_travel_dataset.csv')
df_50k = pd.read_csv(f'{DATA_DIR}/travel_dataset_50k.csv')

print(f"Dataset original: {len(df_original)} lignes")
print(f"Dataset étendu: {len(df_extended)} lignes")
print(f"Dataset 50k: {len(df_50k)} lignes")

# Aperçu des données
print("\n=== Aperçu du dataset original ===")
print(df_original.head())
print("\n=== Info du dataset ===")
print(df_original.info())


### 2.2 Analyse de la Distribution des Classes


In [ ]:
# Distribution des classes dans chaque dataset
print("=== Distribution des classes ===")
for name, df in [('Original', df_original), ('Extended', df_extended), ('50k', df_50k)]:
    if 'VALIDITY' in df.columns:
        print(f"\n{name}:")
        print(df['VALIDITY'].value_counts())
        print(f"Pourcentages:")
        print(df['VALIDITY'].value_counts(normalize=True) * 100)


## 3. Phase 2: Nettoyage & Préparation des Données

### 3.1 Consolidation des Datasets


In [ ]:
# Fusionner tous les datasets
def consolidate_datasets(dfs):
    """Fusionne plusieurs dataframes en un seul"""
    # Uniformiser les noms de colonnes
    for df in dfs:
        if 'Sentence' not in df.columns and 'sentence' in df.columns:
            df.rename(columns={'sentence': 'Sentence'}, inplace=True)
        if 'VALIDITY' not in df.columns and 'validity' in df.columns:
            df.rename(columns={'validity': 'VALIDITY'}, inplace=True)
    
    # Fusionner
    combined = pd.concat(dfs, ignore_index=True)
    
    # Supprimer les doublons exacts
    combined = combined.drop_duplicates(subset=['Sentence'], keep='first')
    
    return combined

# Appliquer
df_combined = consolidate_datasets([df_original, df_extended, df_50k])
print(f"Dataset consolidé: {len(df_combined)} phrases uniques")
print(f"\nDistribution:")
print(df_combined['VALIDITY'].value_counts())


## 4. Phase 3: Feature Engineering & Vectorisation


In [ ]:
# Filtrer pour ne garder que VALID et INVALID
df_filtered = df_combined[df_combined['VALIDITY'].isin(['VALID', 'INVALID'])].copy()

# Préparer les données
X_text = df_filtered['Sentence'].values
y = df_filtered['VALIDITY'].values

# Vectorisation TF-IDF
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),  # unigrams et bigrams
    min_df=2,
    max_df=0.95,
    lowercase=True
)

X_vectorized = vectorizer.fit_transform(X_text)
print(f"Shape de la matrice vectorisée: {X_vectorized.shape}")


## 5. Phase 4: Baseline Model - Régression Logistique

### 5.1 Split Train/Validation/Test


In [ ]:
# Split train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

# Split train/validation (80/20 du train)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f"Train: {X_train.shape[0]} échantillons")
print(f"Validation: {X_val.shape[0]} échantillons")
print(f"Test: {X_test.shape[0]} échantillons")


In [ ]:
# Initialiser et entraîner le modèle
lr_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced'
)

print("Entraînement du modèle...")
lr_model.fit(X_train, y_train)
print("✅ Modèle entraîné")

# Prédictions et évaluation
y_test_pred = lr_model.predict(X_test)

print("\n=== Métriques sur Test Set ===")
print(f"Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

# Matrice de confusion
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=lr_model.classes_, 
            yticklabels=lr_model.classes_)
plt.title('Matrice de Confusion - Test Set')
plt.ylabel('Vrai')
plt.xlabel('Prédit')
plt.show()


## 6. Conclusions & Prochaines Étapes

### Résultats du Baseline
- **Accuracy:** [à remplir après exécution]
- **F1-Score VALID:** [à remplir]
- **F1-Score INVALID:** [à remplir]

### Observations
- [ ] Points forts du modèle
- [ ] Points faibles identifiés
- [ ] Patterns d'erreurs récurrents

### Prochaines Étapes
1. Améliorer le dataset avec les cas problématiques
2. Expérimenter avec d'autres features
3. Tester des modèles plus avancés (transformers)
